<sub>**The Annotated ML Handbook** · Part I: Python · by Mahir Absar Khan</sub>

# Chapter 2: Data structures: lists, tuples, dictionaries, sets

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mahir-606/annotated-ml-handbook/blob/main/notebooks/part-1-python/02-python-collections.ipynb) &nbsp; [📚 Contents](../../book/README.md) &nbsp; [🌐 Web edition](https://mahir-606.github.io/annotated-ml-handbook/)

*Run the setup cell first, then read and run the cells from top to bottom. Every code cell is followed by a table explaining what each line does.*

In [1]:
# Repository setup (not part of the book's examples): run this cell first.
import os, sys
from pathlib import Path

# Files the examples create (CSV, models, figures) go into ./outputs next to the notebook.
OUT = Path.cwd() if Path.cwd().name == "outputs" else Path.cwd() / "outputs"
OUT.mkdir(exist_ok=True)
os.chdir(OUT)

# insurance.csv is the practice dataset created in Chapter 11 and reused by later chapters.
# If it is not here yet, copy it from the repository's data/ folder (or download it in Colab).
if not Path("insurance.csv").exists():
    import shutil, urllib.request
    for base in OUT.parents:
        if (base / "data" / "insurance.csv").exists():
            shutil.copy(base / "data" / "insurance.csv", "insurance.csv")
            break
    else:
        try:
            urllib.request.urlretrieve("https://raw.githubusercontent.com/Mahir-606/"
                                       "annotated-ml-handbook/main/data/insurance.csv", "insurance.csv")
        except Exception as e:
            print("Could not fetch insurance.csv; run Chapter 11 first.", e)

# The figure style used in the book.
import matplotlib.pyplot as plt
from cycler import cycler
plt.rcParams.update({
    "figure.figsize": (7.2, 4.0), "figure.dpi": 100,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.28, "grid.linewidth": 0.6,
    "axes.prop_cycle": cycler(color=["#1f4fff", "#e0245e", "#0f9b62", "#ff6a00",
                                     "#7b3ff2", "#0091b3", "#d99a00", "#d42a2a"]),
    "axes.titleweight": "bold", "axes.titlesize": 12.5, "axes.titlelocation": "left",
    "font.size": 10.5, "legend.frameon": False, "image.cmap": "viridis",
})

# Plotly: interactive in Jupyter/Colab; tools/run_all.py stores static PNGs so GitHub can show them.
try:
    import plotly.io as pio
    pio.renderers.default = os.environ.get("PLOTLY_RENDERER") or (
        "colab" if "google.colab" in sys.modules else "notebook_connected")
except ImportError:
    pass

Real data comes in groups: a column of measurements, a record of a patient, a set of unique labels. Python's four built-in collections cover nearly every case, and understanding them makes NumPy and pandas feel familiar later.

| Type | Literal | Ordered | Mutable | Duplicates | Typical use |
|---|---|---|---|---|---|
| `list` | `[1, 2, 3]` | yes | yes | yes | a sequence you will grow, sort or edit |
| `tuple` | `(1, 2, 3)` | yes | no | yes | a fixed record, e.g. `(lat, lon)`, a shape `(rows, cols)` |
| `dict` | `{"a": 1}` | insertion order | yes | keys unique | lookup by name: configs, hyperparameters, JSON |
| `set` | `{1, 2, 3}` | no | yes | no | uniqueness, membership tests, set algebra |

## 2.1 Lists · *Beginner*

> **📖 Definitions**
>
> **List**: an ordered, **mutable** collection of items in square brackets. Items can be of any type and can repeat. *Example:* `scores = [88, 92, 79]`.
>
> **Index**: the position of an item in a sequence, counting from 0 at the left. Negative indices count from the right: `x[-1]` is the last item.
>
> **Mutable / immutable**: a mutable object can be changed after it is created (lists, dicts, sets, NumPy arrays, DataFrames); an immutable one cannot (numbers, strings, tuples).

**Creating, indexing and modifying a list**

In [2]:
scores = [88, 92, 79, 95]
print(scores[0], scores[-1])
scores[2] = 81
scores.append(70)
scores.extend([99, 60])
scores.insert(1, 100)
print(scores)
last = scores.pop()
scores.remove(100)
print(last, scores)
print(len(scores), sum(scores), max(scores), 95 in scores)

88 95
[88, 100, 92, 81, 95, 70, 99, 60]
60 [88, 92, 81, 95, 70, 99]
6 525 99 True


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>scores = [88, 92, 79, 95]</code></td><td>Square brackets create a list. Items can be any type, even mixed.</td></tr>
<tr><td><code>print(scores[0], scores[-1])</code></td><td>Indexing works exactly like strings: first and last item.</td></tr>
<tr><td><code>scores[2] = 81</code></td><td>Lists are <i>mutable</i>: you can replace an item in place.</td></tr>
<tr><td><code>scores.append(70)</code></td><td>Add one item to the end.</td></tr>
<tr><td><code>scores.extend([99, 60])</code></td><td>Add every item from another iterable to the end.</td></tr>
<tr><td><code>scores.insert(1, 100)</code></td><td>Insert 100 at index 1, shifting the rest right.</td></tr>
<tr><td><code>print(scores)</code></td><td>Display the value of <code>scores</code>.</td></tr>
<tr><td><code>last = scores.pop()</code></td><td>Remove AND return the last item (60). <code>pop(i)</code> removes index i.</td></tr>
<tr><td><code>scores.remove(100)</code></td><td>Remove the first item equal to 100 (by value, not index).</td></tr>
<tr><td><code>print(last, scores)</code></td><td>Show the popped value (60) and the list without 100 and 60.</td></tr>
<tr><td><code>print(len(scores), sum(scores), max(scores), 95 in scores)</code></td><td>Built-ins work on any list of numbers.</td></tr>
</table>
</details>

### 2.1.1 Slicing · *Beginner*

> **📖 Definitions**
>
> **Slice**: a sub-sequence selected with `[start:stop:step]`. `start` is included and `stop` is excluded. *Example:* `x[1:4]` gives items 1, 2 and 3.


Slicing `[start:stop:step]` returns a new list. The same syntax powers NumPy arrays and pandas, so it is worth mastering now.

**Slices**

In [3]:
nums = list(range(10))
print(nums[2:5])
print(nums[:3], nums[-3:])
print(nums[::2])
print(nums[::-1])
nums[1:3] = [10, 20]
print(nums)

[2, 3, 4]
[0, 1, 2] [7, 8, 9]
[0, 2, 4, 6, 8]
[9, 8, 7, 6, 5, 4, 3, 2, 1, 0]
[0, 10, 20, 3, 4, 5, 6, 7, 8, 9]


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>nums = list(range(10))</code></td><td><code>range(10)</code> produces 0..9; <code>list()</code> materialises it: [0, 1, ..., 9].</td></tr>
<tr><td><code>print(nums[2:5])</code></td><td>Items at index 2, 3, 4 (stop is excluded).</td></tr>
<tr><td><code>print(nums[:3], nums[-3:])</code></td><td>First three, last three.</td></tr>
<tr><td><code>print(nums[::2])</code></td><td>Every second item: step = 2.</td></tr>
<tr><td><code>print(nums[::-1])</code></td><td>Reversed copy.</td></tr>
<tr><td><code>nums[1:3] = [10, 20]</code></td><td>Slice assignment replaces a section in place.</td></tr>
<tr><td><code>print(nums)</code></td><td>Display the value of <code>nums</code>.</td></tr>
</table>
</details>

### 2.1.2 Sorting · *Beginner*

**sorted() vs .sort()**

In [4]:
names = ["mia", "Ola", "zed", "ben"]
print(sorted(names))
print(sorted(names, key=str.lower))
print(sorted(names, key=len, reverse=True))
names.sort()
print(names)

['Ola', 'ben', 'mia', 'zed']
['ben', 'mia', 'Ola', 'zed']
['mia', 'Ola', 'zed', 'ben']
['Ola', 'ben', 'mia', 'zed']


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>names = [&quot;mia&quot;, &quot;Ola&quot;, &quot;zed&quot;, &quot;ben&quot;]</code></td><td>A list of names with mixed capitalisation.</td></tr>
<tr><td><code>print(sorted(names))</code></td><td><code>sorted()</code> returns a NEW sorted list. Uppercase sorts before lowercase!</td></tr>
<tr><td><code>print(sorted(names, key=str.lower))</code></td><td><code>key</code> = function applied to each item before comparing: case-insensitive.</td></tr>
<tr><td><code>print(sorted(names, key=len, reverse=True))</code></td><td>Sort by length, longest first.</td></tr>
<tr><td><code>names.sort()</code></td><td><code>.sort()</code> sorts IN PLACE and returns None.</td></tr>
<tr><td><code>print(names)</code></td><td>Display the value of <code>names</code>.</td></tr>
</table>
</details>

> **⚠️ Pitfall** · **`x = x.sort()` destroys your list**
>
> Because `.sort()` returns `None`, writing `scores = scores.sort()` replaces your data with `None`. Use either `scores.sort()` on its own line or `scores = sorted(scores)`.

## 2.2 Tuples · *Beginner*

> **📖 Definitions**
>
> **Tuple**: an ordered, **immutable** collection in round brackets. Use it for fixed groups of values. *Example:* `point = (3.0, 4.0)`, an array's `shape` such as `(100, 4)`.
>
> **Unpacking**: assigning the items of a sequence to several names at once. *Example:* `x, y = point`.


A tuple is an immutable list. Use it for things that belong together and should not change, such as coordinates, RGB colours, or the `shape` of an array.

**Tuples and unpacking**

In [5]:
point = (3.0, 4.0)
x, y = point
print(x, y, len(point))
single = (5,)
print(type(single), type((5)))
a, b = 1, 2
a, b = b, a
print(a, b)
first, *rest = [10, 20, 30, 40]
print(first, rest)

3.0 4.0 2
<class 'tuple'> <class 'int'>
2 1
10 [20, 30, 40]


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>point = (3.0, 4.0)</code></td><td>Parentheses create a tuple; actually the comma does: <code>3.0, 4.0</code> is also a tuple.</td></tr>
<tr><td><code>x, y = point</code></td><td><i>Unpacking</i>: assign each element to its own name.</td></tr>
<tr><td><code>print(x, y, len(point))</code></td><td>Both coordinates, and the tuple&#x27;s length (2).</td></tr>
<tr><td><code>single = (5,)</code></td><td>A one-element tuple needs a trailing comma; <code>(5)</code> is just the number 5.</td></tr>
<tr><td><code>print(type(single), type((5)))</code></td><td>Shows <code>tuple</code> for <code>(5,)</code> but <code>int</code> for <code>(5)</code>.</td></tr>
<tr><td><code>a, b = 1, 2</code></td><td>Assign two names at once.</td></tr>
<tr><td><code>a, b = b, a</code></td><td>Swap two variables without a temporary: builds a tuple, then unpacks it.</td></tr>
<tr><td><code>print(a, b)</code></td><td>Now <code>a</code> is 2 and <code>b</code> is 1.</td></tr>
<tr><td><code>first, *rest = [10, 20, 30, 40]</code></td><td>Star-unpacking collects &quot;everything else&quot; into a list.</td></tr>
<tr><td><code>print(first, rest)</code></td><td><code>first</code> is 10; <code>rest</code> is the list [20, 30, 40].</td></tr>
</table>
</details>

**Tuples cannot be changed**

⚠️ *This cell raises an error **on purpose**, so you can see the real message.*

In [6]:
shape = (28, 28)
shape[0] = 32

TypeError: 'tuple' object does not support item assignment

<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>shape = (28, 28)</code></td><td>E.g. the height and width of an image.</td></tr>
<tr><td><code>shape[0] = 32</code></td><td>Item assignment on a tuple raises TypeError. Create a new tuple instead.</td></tr>
</table>
</details>

## 2.3 Dictionaries · *Beginner*

> **📖 Definitions**
>
> **Dictionary (dict)**: a mutable collection of **key–value pairs** in curly braces, for looking up a value by its key. Keys must be unique and immutable. *Example:* `params = {"C": 1.0, "kernel": "rbf"}`; `params["C"]` gives `1.0`.
>
> **Key and value**: in a dictionary, the key is the label you look up (often a string) and the value is the data stored under it.


A dictionary maps **keys** to **values**. Lookup by key is extremely fast regardless of size. Keys must be immutable (strings, numbers, tuples).

**Building and querying a dictionary**

In [7]:
params = {"model": "svm", "C": 1.0, "kernel": "rbf"}
print(params["C"])
params["gamma"] = 0.1
params["C"] = 10.0
print(params.get("degree"))
print(params.get("degree", 3))
print("kernel" in params)
del params["gamma"]
for key, value in params.items():
    print(f"{key:>7} -> {value}")
print(list(params.keys()), list(params.values()))

1.0
None
3
True
  model -> svm
      C -> 10.0
 kernel -> rbf
['model', 'C', 'kernel'] ['svm', 10.0, 'rbf']


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>params = {&quot;model&quot;: &quot;svm&quot;, &quot;C&quot;: 1.0, &quot;kernel&quot;: &quot;rbf&quot;}</code></td><td><code>key: value</code> pairs inside curly braces.</td></tr>
<tr><td><code>print(params[&quot;C&quot;])</code></td><td>Look up a value by its key.</td></tr>
<tr><td><code>params[&quot;gamma&quot;] = 0.1</code></td><td>Add a new key, or overwrite an existing one.</td></tr>
<tr><td><code>params[&quot;C&quot;] = 10.0</code></td><td>Assigning to an existing key overwrites its value.</td></tr>
<tr><td><code>print(params.get(&quot;degree&quot;))</code></td><td><code>.get</code> returns None instead of crashing if the key is missing...</td></tr>
<tr><td><code>print(params.get(&quot;degree&quot;, 3))</code></td><td>...or a default value you supply.</td></tr>
<tr><td><code>print(&quot;kernel&quot; in params)</code></td><td><code>in</code> checks KEYS, not values.</td></tr>
<tr><td><code>del params[&quot;gamma&quot;]</code></td><td>Remove a key.</td></tr>
<tr><td><code>for key, value in params.items():</code></td><td><code>.items()</code> yields (key, value) pairs: the most common loop.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;print(f&quot;{key:&gt;7} -&gt; {value}&quot;)</code></td><td>Right-align the key in 7 characters, then show its value.</td></tr>
<tr><td><code>print(list(params.keys()), list(params.values()))</code></td><td>All keys and all values, converted to lists for printing.</td></tr>
</table>
</details>

**The KeyError**

⚠️ *This cell raises an error **on purpose**, so you can see the real message.*

In [8]:
params = {"model": "svm"}
print(params["learning_rate"])

KeyError: 'learning_rate'

<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>params = {&quot;model&quot;: &quot;svm&quot;}</code></td><td>A dictionary with only one key.</td></tr>
<tr><td><code>print(params[&quot;learning_rate&quot;])</code></td><td>A missing key raises KeyError. Use <code>.get()</code> or check <code>in</code> first.</td></tr>
</table>
</details>

### 2.3.1 Nested structures · *Intermediate*

Dictionaries and lists nest freely. This is exactly the shape of JSON data and of many configuration files.

**A nested record**

In [9]:
experiment = {
    "name": "baseline",
    "metrics": {"acc": 0.91, "f1": 0.88},
    "seeds": [0, 1, 2],
}
print(experiment["metrics"]["f1"])
experiment["seeds"].append(3)
print(experiment["seeds"])
merged = {**experiment["metrics"], "auc": 0.95}
print(merged)
print(experiment["metrics"] | {"acc": 0.93})

0.88
[0, 1, 2, 3]
{'acc': 0.91, 'f1': 0.88, 'auc': 0.95}
{'acc': 0.93, 'f1': 0.88}


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>experiment = {</code></td><td>Start a dictionary that spans several lines.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&quot;name&quot;: &quot;baseline&quot;,</code></td><td>A plain string value.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&quot;metrics&quot;: {&quot;acc&quot;: 0.91, &quot;f1&quot;: 0.88},</code></td><td>A dict inside a dict.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&quot;seeds&quot;: [0, 1, 2],</code></td><td>A list inside a dict.</td></tr>
<tr><td><code>}</code></td><td>Close the dictionary opened above.</td></tr>
<tr><td><code>print(experiment[&quot;metrics&quot;][&quot;f1&quot;])</code></td><td>Chain lookups from the outside in.</td></tr>
<tr><td><code>experiment[&quot;seeds&quot;].append(3)</code></td><td>Modify the inner list directly.</td></tr>
<tr><td><code>print(experiment[&quot;seeds&quot;])</code></td><td>Display the value of <code>experiment[&quot;seeds&quot;]</code>.</td></tr>
<tr><td><code>merged = {**experiment[&quot;metrics&quot;], &quot;auc&quot;: 0.95}</code></td><td><code>**</code> unpacks a dict into a new one; then add a key.</td></tr>
<tr><td><code>print(merged)</code></td><td>Display the value of <code>merged</code>.</td></tr>
<tr><td><code>print(experiment[&quot;metrics&quot;] | {&quot;acc&quot;: 0.93})</code></td><td><code>|</code> merges dicts (right side wins on clashes). Python 3.9+.</td></tr>
</table>
</details>

## 2.4 Sets · *Beginner*

> **📖 Definitions**
>
> **Set**: an unordered, mutable collection of **unique** items in curly braces, with fast membership tests and set algebra (union, intersection, difference). *Example:* `{"cat", "dog"}`.

**Uniqueness and set algebra**

In [10]:
labels = ["cat", "dog", "cat", "bird", "dog"]
unique = set(labels)
print(unique, len(unique))
a = {1, 2, 3, 4}
b = {3, 4, 5}
print(a | b, a & b, a - b, a ^ b)
print(3 in a)
train_ids, test_ids = {1, 2, 3}, {3, 4}
print("Overlap:", train_ids & test_ids)

{'bird', 'dog', 'cat'} 3
{1, 2, 3, 4, 5} {3, 4} {1, 2} {1, 2, 5}
True
Overlap: {3}


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>labels = [&quot;cat&quot;, &quot;dog&quot;, &quot;cat&quot;, &quot;bird&quot;, &quot;dog&quot;]</code></td><td>A list with repeated labels.</td></tr>
<tr><td><code>unique = set(labels)</code></td><td>Converting to a set drops duplicates. Order is NOT preserved.</td></tr>
<tr><td><code>print(unique, len(unique))</code></td><td>Three unique labels remain.</td></tr>
<tr><td><code>a = {1, 2, 3, 4}</code></td><td>A set literal: curly braces with single values (no colons).</td></tr>
<tr><td><code>b = {3, 4, 5}</code></td><td>A second set that overlaps the first in 3 and 4.</td></tr>
<tr><td><code>print(a | b, a &amp; b, a - b, a ^ b)</code></td><td>Union, intersection, difference, symmetric difference.</td></tr>
<tr><td><code>print(3 in a)</code></td><td>Membership tests on sets are very fast even for millions of items.</td></tr>
<tr><td><code>train_ids, test_ids = {1, 2, 3}, {3, 4}</code></td><td>Sample IDs in a training set and a test set.</td></tr>
<tr><td><code>print(&quot;Overlap:&quot;, train_ids &amp; test_ids)</code></td><td>A real use: detect samples leaking between train and test sets!</td></tr>
</table>
</details>

> **⚠️ Pitfall** · **`{}` is an empty dict, not a set**
>
> Write `set()` for an empty set. `{}` creates an empty dictionary.

## 2.5 Comprehensions · *Beginner*

> **📖 Definitions**
>
> **Comprehension**: a compact expression that builds a new list, dict or set from an iterable, optionally with a filter. *Example:* `[x ** 2 for x in nums if x > 0]`.


A comprehension builds a new collection from an existing one in a single readable expression. They replace most simple `for` loops that build lists.

**List, dict and set comprehensions**

In [11]:
nums = [1, 2, 3, 4, 5, 6]
squares = [n ** 2 for n in nums]
evens = [n for n in nums if n % 2 == 0]
labels = ["even" if n % 2 == 0 else "odd" for n in nums]
print(squares, evens, labels, sep="\n")
word_len = {w: len(w) for w in ["numpy", "pandas", "keras"]}
print(word_len)
first_letters = {w[0] for w in ["apple", "avocado", "banana"]}
print(first_letters)
pairs = [(i, j) for i in range(3) for j in range(2)]
print(pairs)

[1, 4, 9, 16, 25, 36]
[2, 4, 6]
['odd', 'even', 'odd', 'even', 'odd', 'even']
{'numpy': 5, 'pandas': 6, 'keras': 5}
{'a', 'b'}
[(0, 0), (0, 1), (1, 0), (1, 1), (2, 0), (2, 1)]


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>nums = [1, 2, 3, 4, 5, 6]</code></td><td>The input list.</td></tr>
<tr><td><code>squares = [n ** 2 for n in nums]</code></td><td>&quot;n squared, for each n in nums&quot;.</td></tr>
<tr><td><code>evens = [n for n in nums if n % 2 == 0]</code></td><td>Add a filter with <code>if</code> at the end.</td></tr>
<tr><td><code>labels = [&quot;even&quot; if n % 2 == 0 else &quot;odd&quot; for n in nums]</code></td><td>An if/else <i>expression</i> goes BEFORE the <code>for</code>.</td></tr>
<tr><td><code>print(squares, evens, labels, sep=&quot;\n&quot;)</code></td><td><code>sep=&quot;\n&quot;</code> prints each argument on its own line.</td></tr>
<tr><td><code>word_len = {w: len(w) for w in [&quot;numpy&quot;, &quot;pandas&quot;, &quot;keras&quot;]}</code></td><td>Dict comprehension: <code>key: value for ...</code>.</td></tr>
<tr><td><code>print(word_len)</code></td><td>Display the value of <code>word_len</code>.</td></tr>
<tr><td><code>first_letters = {w[0] for w in [&quot;apple&quot;, &quot;avocado&quot;, &quot;banana&quot;]}</code></td><td>Set comprehension: unique first letters.</td></tr>
<tr><td><code>print(first_letters)</code></td><td>Display the value of <code>first_letters</code>.</td></tr>
<tr><td><code>pairs = [(i, j) for i in range(3) for j in range(2)]</code></td><td>Two <code>for</code>s = nested loops (outer loop first).</td></tr>
<tr><td><code>print(pairs)</code></td><td>Display the value of <code>pairs</code>.</td></tr>
</table>
</details>

## 2.6 Mutability, references and copies · *Intermediate*

> **📖 Definitions**
>
> **Reference**: a name pointing to an object. Assignment (`b = a`) creates a second reference to the same object, not a copy.
>
> **Shallow copy / deep copy**: a shallow copy duplicates the outer container but shares the objects inside it; a deep copy (`copy.deepcopy`) duplicates everything, recursively.


This is the single most common source of baffling bugs. **Assignment never copies.** It makes a second name for the same object.

**Two names, one list**

In [12]:
a = [1, 2, 3]
b = a
b.append(4)
print(a)
print(a is b)
c = a.copy()
c.append(5)
print(a, c, a is c)

[1, 2, 3, 4]
True
[1, 2, 3, 4] [1, 2, 3, 4, 5] False


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>a = [1, 2, 3]</code></td><td>Create a list and name it <code>a</code>.</td></tr>
<tr><td><code>b = a</code></td><td><code>b</code> is NOT a copy. Both names point at the same list.</td></tr>
<tr><td><code>b.append(4)</code></td><td>Modifying through <code>b</code>...</td></tr>
<tr><td><code>print(a)</code></td><td>...changes what <code>a</code> sees, because it is the same object.</td></tr>
<tr><td><code>print(a is b)</code></td><td><code>is</code> confirms they are identical objects.</td></tr>
<tr><td><code>c = a.copy()</code></td><td>A real (shallow) copy. Also: <code>list(a)</code> or <code>a[:]</code>.</td></tr>
<tr><td><code>c.append(5)</code></td><td>Change only the copy.</td></tr>
<tr><td><code>print(a, c, a is c)</code></td><td>Now <code>a</code> is unaffected.</td></tr>
</table>
</details>

**Shallow vs deep copies of nested data**

In [13]:
import copy
grid = [[0, 0], [0, 0]]
shallow = grid.copy()
deep = copy.deepcopy(grid)
grid[0][0] = 99
print(shallow[0][0], deep[0][0])
bad = [[0] * 2] * 2
bad[0][0] = 1
print(bad)
good = [[0] * 2 for _ in range(2)]
good[0][0] = 1
print(good)

99 0
[[1, 0], [1, 0]]
[[1, 0], [0, 0]]


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>import copy</code></td><td>Standard-library module for copying.</td></tr>
<tr><td><code>grid = [[0, 0], [0, 0]]</code></td><td>A list of two lists: a 2x2 grid.</td></tr>
<tr><td><code>shallow = grid.copy()</code></td><td>Copies the OUTER list only; the inner lists are still shared.</td></tr>
<tr><td><code>deep = copy.deepcopy(grid)</code></td><td>Recursively copies everything.</td></tr>
<tr><td><code>grid[0][0] = 99</code></td><td>Change an inner list.</td></tr>
<tr><td><code>print(shallow[0][0], deep[0][0])</code></td><td>The shallow copy sees 99; the deep copy still has 0.</td></tr>
<tr><td><code>bad = [[0] * 2] * 2</code></td><td>Classic trap: <code>* 2</code> repeats a reference to the SAME inner list.</td></tr>
<tr><td><code>bad[0][0] = 1</code></td><td>Change the first element of the first row...</td></tr>
<tr><td><code>print(bad)</code></td><td>Both rows changed! Build grids with a comprehension instead:</td></tr>
<tr><td><code>good = [[0] * 2 for _ in range(2)]</code></td><td>Each iteration creates a fresh inner list.</td></tr>
<tr><td><code>good[0][0] = 1</code></td><td>Change one cell: only the first row is affected.</td></tr>
<tr><td><code>print(good)</code></td><td>Display the value of <code>good</code>.</td></tr>
</table>
</details>

> **🐞 Bug hunt** · **"I changed one variable and another one changed too"**
>
> You have two names for one mutable object (a list, dict, NumPy array or DataFrame). Make an explicit copy: `.copy()` for lists, dicts, NumPy arrays and DataFrames, or `copy.deepcopy()` for nested structures.

> **✍️ Try it yourself**
>
> 1. From `words = ["Data", "science", "IS", "fun", "data"]`, build the set of lowercase unique words.
> 2. Build a dict mapping each number 1–10 to `"fizz"` if divisible by 3, else the number itself.
> 3. Explain why `t = ([1, 2], 3); t[0].append(9)` works even though tuples are immutable.
>
> *Worked solutions:* [solutions-parts-1-4.ipynb](../../solutions/solutions-parts-1-4.ipynb), section *Chapter 2*.

---

⬅️ [Chapter 1: Values, variables and types](01-python-basics.ipynb) · [📚 Contents](../../book/README.md) · [Chapter 3: Control flow: decisions and loops](03-python-control.ipynb) ➡️